# Lab 5: Incremental Learning -- Models That Update in Real Time

## Goal

- Understand the difference between batch and online learning,
- Implement SGD-based incremental model,
- Build a consumer that learns AND scores simultaneously.

**Business case (continued):** Your static model from Lab 4 gets stale over time. New fraud patterns emerge. You need a model that adapts continuously.

---

## Part 1: Batch vs Online Learning

### Task 1.1 -- Batch model (recap)

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Reuse data from Lab 4
np.random.seed(42)
n = 1000
X = np.column_stack([
    np.random.lognormal(5, 1, n).clip(5, 5000),
    np.random.normal(14, 4, n).clip(0, 23),
    np.random.binomial(1, 0.3, n),
    np.random.poisson(3, n),
])
y = (X[:, 0] > 2000).astype(int)  # simplified fraud label

# Batch: train on all data at once
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

batch_model = SGDClassifier(loss='log_loss', random_state=42)
batch_model.fit(X_scaled, y)
print(f"Batch accuracy: {batch_model.score(X_scaled, y):.3f}")

### Task 1.2 -- Online learning with partial_fit

Train the same model incrementally -- one sample at a time.

In [ ]:
online_model = SGDClassifier(loss='log_loss', random_state=42)

# We must call partial_fit with classes on first call
online_model.partial_fit(X_scaled[:1], y[:1], classes=[0, 1])

# YOUR CODE
# Loop through samples one by one (or in mini-batches of 10)
# Use online_model.partial_fit(X_batch, y_batch)
# Every 100 samples, print current accuracy on ALL data
# Compare convergence speed with batch model

### Task 1.3 -- Simulated concept drift

After 500 samples, change the fraud pattern (e.g., lower the threshold).
Observe how the online model adapts while a batch model would not.

In [ ]:
# YOUR CODE
# 1. Train online model on first 500 samples (normal pattern)
# 2. Generate 500 new samples with DIFFERENT fraud pattern
#    (e.g., fraud if amount > 1000 instead of > 2000)
# 3. Continue partial_fit on new samples
# 4. Track accuracy over time -- does the model adapt?

---

## Part 2: Online Learning with Kafka

### Task 2.1 -- Learning consumer

Write a Kafka consumer that:
1. Reads labeled transactions (add a `fraud` field to your producer)
2. Scores each transaction with the current model
3. Updates the model with `partial_fit` using the true label
4. Prints rolling accuracy (last 100 predictions)

In [ ]:
%%file learning_consumer.py
from kafka import KafkaConsumer
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
import json
import numpy as np
from collections import deque

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='learning-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

model = SGDClassifier(loss='log_loss')
scaler = StandardScaler()
initialized = False
recent_correct = deque(maxlen=100)

# YOUR CODE
# For each message:
# 1. Extract features: [amount, hour, is_electronics_flag, ...]
# 2. If not initialized: partial_fit with classes=[0,1] and fit scaler
# 3. Score with current model (before updating!)
# 4. partial_fit with true label
# 5. Track accuracy in recent_correct
# 6. Every 20 messages: print rolling accuracy

---

## Homework

1. Modify the producer to simulate **concept drift** at message #200 (change fraud pattern).
2. Compare online model accuracy before and after the drift.
3. Combine Lab 3 rules + Lab 5 online model: use rules as features for the online model.
4. Push to Git.

**Next lab:** Exploratory data analysis on historical transaction data with pandas.